# Conversion de GSE149689 en AnnData

Ce notebook télécharge uniquement les données d'expression traitées de GEO et les convertit en fichier `.h5ad`, sans analyse biologique.

## 1. Ressources disponibles

In [ ]:
!free -h

## 2. Installation des dépendances manquantes

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = ["scanpy", "anndata"]
missing_packages = [
    package for package in required_packages
    if importlib.util.find_spec(package) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("scanpy et anndata sont déjà disponibles.")

## 3. Imports et chemins

In [ ]:
from pathlib import Path
import gzip
import shutil
import urllib.request

import anndata as ad
import scanpy as sc

data_dir = Path("/content/GSE149689")
output_path = Path("/content/GSE149689_raw.h5ad")
data_dir.mkdir(parents=True, exist_ok=True)

## 4. Téléchargement des fichiers traités GEO

In [ ]:
base_url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE149nnn/GSE149689/suppl"
filenames = [
    "GSE149689_matrix.mtx.gz",
    "GSE149689_barcodes.tsv.gz",
    "GSE149689_features.tsv.gz",
]

for filename in filenames:
    destination = data_dir / filename
    if destination.exists():
        print(f"Déjà présent : {destination}")
    else:
        print(f"Téléchargement : {filename}")
        urllib.request.urlretrieve(f"{base_url}/{filename}", destination)

## 5. Vérification des fichiers et des dimensions annoncées

In [ ]:
for path in sorted(data_dir.iterdir()):
    if path.is_file():
        print(f"{path.name}: {path.stat().st_size:,} octets")

matrix_path = data_dir / "GSE149689_matrix.mtx.gz"
with gzip.open(matrix_path, "rt") as handle:
    for line in handle:
        if not line.startswith("%"):
            n_features, n_barcodes, n_nonzero = map(int, line.split())
            break

print(f"Dimensions Matrix Market annoncées : {n_features} features × {n_barcodes} barcodes")
print(f"Valeurs non nulles annoncées : {n_nonzero:,}")

## 6. Chargement sparse dans AnnData

In [ ]:
adata = sc.read_10x_mtx(
    data_dir,
    var_names="gene_symbols",
    make_unique=False,
    prefix="GSE149689_",
)

## 7. Inspection minimale et unicité des noms

In [ ]:
print(adata)
print(adata.shape)
print(type(adata.X))
print(adata.X.dtype)

print("Noms de cellules uniques :", adata.obs_names.is_unique)
print("Noms de gènes uniques avant correction :", adata.var_names.is_unique)

duplicated_gene_names = int(adata.var_names.duplicated().sum())
print("Noms de gènes dupliqués :", duplicated_gene_names)

if duplicated_gene_names:
    adata.var_names_make_unique()

print("Noms de gènes uniques après correction :", adata.var_names.is_unique)

## 8. Sauvegarde du fichier AnnData brut

In [ ]:
adata.write_h5ad(output_path)
print(f"Fichier créé : {output_path}")
print(f"Taille finale : {output_path.stat().st_size / (1024 ** 2):.2f} MiB")

## 9. Récupération du fichier

In [ ]:
drive_directory = Path("/content/drive/MyDrive")

if drive_directory.is_dir():
    drive_path = drive_directory / output_path.name
    shutil.copy2(output_path, drive_path)
    print(f"Copié vers Google Drive : {drive_path}")
else:
    from google.colab import files
    files.download(str(output_path))